# Scheduler et arrêt anticipé : deux décisions séparées

Faire évoluer le taux sans confondre ce réglage avec le moment où arrêter.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#scheduler-et-arret-anticipe).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Une fois la boucle comprise, nous pouvons modifier son déroulement. Un scheduler change le taux d’apprentissage ; un arrêt anticipé décide de ne plus continuer les époques.

Pour voir les deux mécanismes séparément, nous choisissons une suite fictive de pertes de validation : 0,90 ; 0,60 ; 0,62 ; 0,65 ; 0,70. Elles ne viennent pas du petit poids entraîné ci-dessous.

Cette liste volontairement imposée nous assure deux époques sans amélioration après le meilleur point. Dans une vraie boucle, validation serait la mesure calculée sur les données réservées.

| Époque | Validation choisie | Meilleure valeur | Attente |
| --- | --- | --- | --- |
| 1 | 0,90 | 0,90 | 0 |
| 2 | 0,60 | 0,60 | 0 |
| 3 | 0,62 | 0,60 | 1 |
| 4 | 0,65 | 0,60 | 2 → arrêt |

La patience vaut deux époques sans nouvelle meilleure valeur. L’état conservé est celui de l’époque 2.

StepLR reçoit step_size=2 et gamma=0,5 : il multiplie le taux par 0,5 toutes les deux époques. Nous l’appelons après les mises à jour concernées.

## Démonstration 1 · Suivre le taux, l’attente et le poids conservé

Les sorties rendent explicite l’ordre des décisions.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Le taux commence à 0,1. meilleure commence à l’infini pour que la première valeur finie soit retenue ; attente commence à zéro.

In [4]:
poids = torch.tensor(1., requires_grad=True)
optimiseur = torch.optim.SGD([poids], lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimiseur, step_size=2, gamma=0.5)
validations_fictives = [0.9, 0.6, 0.62, 0.65, 0.7]


In [5]:
observer('poids', poids)
observer('scheduler', scheduler)
observer('validations_fictives', validations_fictives)

poids forme () type torch.float32 requires_grad True
  valeurs : 1.0
validations_fictives = [0.9, 0.6, 0.62, 0.65, 0.7]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
meilleure = float("inf")
attente = 0
meilleur_poids = poids.detach().clone()


In [7]:
observer('meilleure', meilleure)
observer('attente', attente)
observer('meilleur_poids', meilleur_poids)

meilleure = inf
attente = 0
meilleur_poids forme () type torch.float32 requires_grad False
  valeurs : 1.0


### Répéter la même étape sur les données

La perte (poids−2)² utilise un gradient réel. taux_utilise est lu avant step pour décrire le taux de cette époque.

Si la validation imposée est meilleure, nous remettons l’attente à zéro et copions les valeurs du poids. La copie ne sera pas modifiée par les pas suivants.

Sinon, attente augmente. scheduler.step prépare le taux de l’époque suivante ; il ne décide pas de l’arrêt.

La patience est atteinte à l’époque 4. Nous restaurons ensuite le poids de l’époque 2, pas le dernier calculé.

In [8]:
for epoque, validation in enumerate(validations_fictives, start=1):
    optimiseur.zero_grad()
    perte = (poids - 2.) ** 2
    perte.backward()
    taux_utilise = optimiseur.param_groups[0]["lr"]
    optimiseur.step()
    if validation < meilleure:
        meilleure = validation
        meilleur_poids = poids.detach().clone()
        attente = 0
    else:
        attente += 1
    print(f"epoque {epoque}: lr={taux_utilise:.2f}, val={validation:.2f}, attente={attente}")
    scheduler.step()
    if attente >= 2:
        print("arret")
        break


epoque 1: lr=0.10, val=0.90, attente=0
epoque 2: lr=0.10, val=0.60, attente=0
epoque 3: lr=0.05, val=0.62, attente=1
epoque 4: lr=0.05, val=0.65, attente=2
arret


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [9]:
with torch.no_grad():
    poids.copy_(meilleur_poids)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [10]:
print(f"meilleur poids restaure : {poids.item():.6f}")

meilleur poids restaure : 1.360000


Le meilleur état dépend du critère de validation ; le scheduler et la patience ont des fonctions indépendantes.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

ReduceLROnPlateau utilise factor=0,5 et patience=0. Après deux mesures identiques, quel taux reste-t-il ?

In [11]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=1.)
exo_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(exo_opt,factor=.5,patience=0)
exo_scheduler.step(1.)
exo_scheduler.step(1.)
exo_taux = ...  # Remplacer les points par votre réponse.
if exo_taux is Ellipsis or exo_taux is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_taux)-exo_opt.param_groups[0]["lr"]) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La première mesure fixe la référence. La suivante ne l’améliore pas ; le taux est multiplié par le facteur.

</details>

### Exercice 2

Un arrêt anticipé autorise deux validations sans amélioration. Faut-il arrêter après les valeurs 0,5 puis 0,6, avec meilleur=0,4 ?

In [12]:
exo_meilleur = .4
exo_attente = 0
for exo_mesure in [.5,.6]:
    if exo_mesure < exo_meilleur:
        exo_meilleur,exo_attente=exo_mesure,0
    else:
        exo_attente+=1
exo_arreter = ...  # Remplacer les points par votre réponse.
if exo_arreter is Ellipsis or exo_arreter is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_arreter is (exo_attente >= 2)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les deux validations dépassent la meilleure perte. L’arrêt est une règle de sélection, pas une mise à jour des poids.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=1.)
exo_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(exo_opt,factor=.5,patience=0)
exo_scheduler.step(1.)
exo_scheduler.step(1.)
exo_taux = 0.5
if exo_taux is Ellipsis or exo_taux is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_taux)-exo_opt.param_groups[0]["lr"]) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_meilleur = .4
exo_attente = 0
for exo_mesure in [.5,.6]:
    if exo_mesure < exo_meilleur:
        exo_meilleur,exo_attente=exo_mesure,0
    else:
        exo_attente+=1
exo_arreter = True
if exo_arreter is Ellipsis or exo_arreter is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_arreter is (exo_attente >= 2)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [13]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()